# Transformation 
- Remove records with null values on a column
- Remove duplicate records
- Remove the duplicate records based on created timestamp
- CAST with current column names
- Write data in delta table


In [0]:
%sql
select * from project_etl.bronze.v_customers where customer_id is not null;

In [0]:
%sql
select distinct * from project_etl.bronze.v_customers where customer_id is not null order by customer_id;

In [0]:
%sql
create or replace temporary view v_customers_distinct as
select distinct * from project_etl.bronze.v_customers where customer_id is not null order by customer_id;

In [0]:
%sql
select customer_id, customer_name ,max(created_timestamp)
as max_timestamp from v_customers_distinct group by customer_id, customer_name;

In [0]:
%sql
with cte_max as 
(select customer_id,  customer_name , max(created_timestamp)
as max_timestamp from v_customers_distinct
group by customer_id, customer_name)
select t.*
 from v_customers_distinct t
 join cte_max c
 on t.customer_id = c.customer_id and t.created_timestamp = c.max_timestamp and t.customer_name = c.customer_name;

In [0]:
%sql
with cte_max as 
(select customer_id,  customer_name , max(created_timestamp)
as max_timestamp from v_customers_distinct
group by customer_id, customer_name)
select CAST(t.created_timestamp AS TIMESTAMP) as created_timestamp, 
       cast(t.date_of_birth AS DATE) as date_of_birth,
       cast(t.member_since AS DATE) as member_since,
       t.city,
       t.customer_id,
       t.customer_name,
       t.email,
       t.telephone,
      t.source_file_path
     --   t.metadata_file
     --  t.metadata_timestamp
     --  --  cast(t.metadata_file AS STRING) as metadata_file 
 from v_customers_distinct t
 join cte_max c
 on t.customer_id = c.customer_id and t.created_timestamp = c.max_timestamp and t.customer_name = c.customer_name;

In [0]:
%sql
create or replace table  project_etl.silver.customers_silver as
with cte_max as 
(select customer_id,  customer_name , max(created_timestamp)
as max_timestamp from v_customers_distinct
group by customer_id, customer_name)
select CAST(t.created_timestamp AS TIMESTAMP) as created_timestamp, 
       cast(t.date_of_birth AS DATE) as date_of_birth,
       cast(t.member_since AS DATE) as member_since,
       t.city,
       t.customer_id,
       t.customer_name,
       t.email,
       t.telephone,
      t.source_file_path
     --   t.metadata_file
     --  t.metadata_timestamp
     --  --  cast(t.metadata_file AS STRING) as metadata_file 
 from v_customers_distinct t
 join cte_max c
 on t.customer_id = c.customer_id and t.created_timestamp = c.max_timestamp and t.customer_name = c.customer_name;


In [0]:
%sql
select * from project_etl.silver.customers_silver;